In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
import numpy as np
from mplsoccer import Pitch, VerticalPitch
from plottable import ColumnDefinition, Table
from plottable.cmap import normed_cmap
from matplotlib.colors import PowerNorm
from matplotlib.lines import Line2D
from matplotlib.lines import Line2D
from mplsoccer import VerticalPitch
import matplotlib.patches as mpatches
import math

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

from great_tables import GT

from extract_carries import extract_carries
import xg_model_baked

from scipy.stats import zscore
from scipy.stats import norm


*Create metrics class*
        
        *Subtask: simple xG model*
        *Subtask: extrapolate carries*

Create Z-Score Creation Class

Create Export Table Class

Forwards
    npXg
    xG assisted (cba figuring out xA fully, will never have to do that in the real world)
    *shots*
    *npGoals*
    *Aerial %*
    *EPV passing*
    *Dribble/Takeon rate*
    *Carrying*
    *Touches in final third*
    Through balls received
    *Through balls played*
    *Fouls won*

Midfielders
    *Progressive pass distance*
    *Progressive passes*
    *Progressive carries*
    *Passes into penalty area*
    *Total tackles*
    *Successful tackles*
    *Tackle %*
    *Pass completion %*
    *Forward passes*
    *Forward pass %*
    *Interceptions (tackles and interceptions?)*
    *Touches*


Defenders
    *Blocked shots*
    *Recoveries*
    *Blocked Crosses*
    *Clearances*

In [3]:
class Metrics:
    def __init__(self, file_path: str, conditions: str = None, players_tablename: str = 'players'):
        """
        file_path is path to DuckDB database. conditions is a string for the
        WHERE clause if a specific range of events is needed.
        """
        con = duckdb.connect(file_path)
        if conditions is None:
            self.events = con.sql("select * from events").df()
        else:
            self.events = con.sql(f"select * from events where {conditions}").df()
        self.events['playerId'] = self.events['playerId'].fillna(-1).astype('int64')  # keep start/end slides for through-ball math, give them an int id
        self.players = con.sql(f"select * from {players_tablename}").df()
        con.close()

        self.events = xg_model_baked.add_xg(self.events)

    def create_metrics(self):
        ev = self.events
        is_shot = ev["type"].isin(["Goal", "SavedShot", "MissedShots", "ShotOnPost"])
        pens = ev[["penaltyScored", "penaltyMissed"]].fillna(False).astype(bool).any(axis=1)

        shots = ev[is_shot]
        shooting_agg = (
            shots.assign(np_shot=(~pens[is_shot]).astype(int),
                         np_goal=((shots["type"] == "Goal") & ~pens[is_shot]).astype(int))
            .groupby("playerId")
            .agg(
                shots=("np_shot", "sum"),
                np_goals=("np_goal", "sum"),
                npxg=("xG", "sum"),
                big_chances_missed=("bigChanceMissed", "sum"),
                big_chances_scored=("bigChanceScored", "sum"),
            )
        )
        shooting_agg["big_chances"] = (
            shooting_agg["big_chances_missed"] + shooting_agg["big_chances_scored"]
        )

        dribbling_agg = (
            ev.groupby("playerId")
            .agg(
                dribbles_lost=("dribbleLost", "sum"),
                dribbles_won=("dribbleWon", "sum"),
            )
        )
        dribbling_agg["dribbles_total"] = (
            dribbling_agg["dribbles_lost"] + dribbling_agg["dribbles_won"]
        )
        dribbling_agg["dribble_success"] = (
            dribbling_agg["dribbles_won"] / dribbling_agg["dribbles_total"].replace(0, np.nan)
        ) * 100

        passing_df = ev[ev["type"] == "Pass"].copy()
        keypass_cols = ["keyPassLong", "keyPassShort", "keyPassCross",
                        "keyPassThroughball", "keyPassOther"]
        passing_df["isKeyPassOP"] = passing_df[keypass_cols].fillna(False).any(axis=1)
        passing_df["isCompleted"] = passing_df["outcomeType"] == "Successful"

        passing_agg_basic = (
            passing_df.groupby("playerId")
            .agg(
                key_passes_open_play=("isKeyPassOP", "sum"),
                through_balls_completed=("passThroughBallAccurate", "sum"),
                passes=("isTouch", "sum"),
                completed_passes=("isCompleted", "sum"),
                pass_epv=("EPV", "sum"),
                forward_passes=("passForward", "sum"),
                big_chances_created=("bigChanceCreated", "sum"),
            )
        )
        passing_agg_basic["completion_percentage"] = (
            passing_agg_basic["completed_passes"]
            / passing_agg_basic["passes"].replace(0, np.nan)
        ) * 100
        passing_agg_basic["forward_pct"] = (
            passing_agg_basic["forward_passes"]
            / passing_agg_basic["passes"].replace(0, np.nan)
        ) * 100

        ev_sorted = ev.sort_values(["matchId", "minute", "second", "id"])
        ev_sorted["prev_type"] = ev_sorted.groupby("matchId")["type"].shift(1)
        for c in ["keyPassCorner", "keyPassFreekick", "keyPassThrowin"]:
            ev_sorted[f"prev_{c}"] = ev_sorted.groupby("matchId")[c].shift(1)

        shot_rows = ev_sorted[ev_sorted["assist_playerId"].notna()].copy()
        shot_rows["assist_set_piece"] = shot_rows[
            ["prev_keyPassCorner", "prev_keyPassFreekick", "prev_keyPassThrowin"]
        ].fillna(False).astype(bool).any(axis=1)

        DEF_ACTIONS = ["Tackle", "Interception", "BallRecovery", "Clearance", "Challenge"]
        def_ft = ev[(ev["type"].isin(DEF_ACTIONS)) & (ev["x"] >= 66.666667)]
        def_ft_agg = (
            def_ft.groupby("playerId")
            .size()
            .rename("def_actions_final_third")
            .to_frame()
        )

        xga_agg = (
            shot_rows.groupby("assist_playerId")["xG_assisted"]
            .sum().rename("xg_assisted")
        )
        xga_op_agg = (
            shot_rows[~shot_rows["assist_set_piece"]]
            .groupby("assist_playerId")["xG_assisted"]
            .sum().rename("xg_assisted_op")
        )
        for s in (xga_agg, xga_op_agg):
            s.index = s.index.astype("int64")
            s.index.name = "playerId"

        adv = ev[(ev["type"] == "Pass") & (ev["outcomeType"] == "Successful")].copy()
        adv["dist_start"] = np.sqrt((100 - adv["x"]) ** 2 + (50 - adv["y"]) ** 2)
        adv["dist_end"] = np.sqrt((100 - adv["endX"]) ** 2 + (50 - adv["endY"]) ** 2)
        adv["progressive_passing_distance"] = (adv["endX"] - adv["x"]).clip(lower=0.0)
        adv["isProgressivePass"] = (adv["x"] > 33.333333) & (
            adv["dist_start"] <= 0.75 * adv["dist_end"]
        )
        adv["penAreaEnd"] = (adv["endX"] >= 83) & (adv["endY"] >= 21) & (adv["endY"] <= 79)

        passing_agg_advanced = (
            adv.groupby("playerId")
            .agg(
                progressive_passing_distance=("progressive_passing_distance", "sum"),
                progressive_passes=("isProgressivePass", "sum"),
                passes_into_pen_area=("penAreaEnd", "sum"),
            )
        )

        defense_agg = (
            ev.groupby("playerId")
            .agg(
                interceptions=("interceptionWon", "sum"),
                tackles_won=("tackleWon", "sum"),
                tackles_lost=("tackleLost", "sum"),
                errors_leading_to_goal=("errorLeadsToGoal", "sum"),
                errors_leading_to_shot=("errorLeadsToShot", "sum"),
                clearances=("clearanceEffective", "sum"),
                recoveries=("ballRecovery", "sum"),
                blocks=("outfielderBlock", "sum"),
                blocked_crosses=("passCrossBlockedDefensive", "sum"),
                aerial_wins=("duelAerialWon", "sum"),
                aerial_losses=("duelAerialLost", "sum"),
                interceptions_in_box=("interceptionIntheBox", "sum"),
            )
        )
        defense_agg["tackles_total"] = defense_agg["tackles_won"] + defense_agg["tackles_lost"]
        defense_agg["tackle_success"] = (
            defense_agg["tackles_won"] / defense_agg["tackles_total"].replace(0, np.nan)
        ) * 100
        defense_agg["aerials"] = defense_agg["aerial_wins"] + defense_agg["aerial_losses"]
        defense_agg["aerial_success"] = (
            defense_agg["aerial_wins"] / defense_agg["aerials"].replace(0, np.nan)
        ) * 100

        DEF_ACTIONS = ["Tackle", "Interception", "BallRecovery", "Clearance", "BlockedPass"]
        def_ft = ev[(ev["type"].isin(DEF_ACTIONS)) & (ev["x"] >= 66.666667)]
        def_ft_agg = (
            def_ft.groupby("playerId")
            .size()
            .rename("def_actions_final_third")
            .to_frame()
        )
        
        touches = ev[ev["isTouch"] == True]
        touches_agg = (
            touches.groupby("playerId")
            .agg(
                touches=("playerId", "count"),
                touches_in_final_third=("finalThird", "sum"),
                fouls_won=("foulGiven", "sum"),
            )
        )

        carries = extract_carries(ev, 2.0)
        carries["progressive_carrying_distance"] = (
            carries["endX"] - carries["startX"]
        ).clip(lower=0.0)
        carries["isProgressive"] = (
            ((carries["startX"] < 60) & (carries["endX"] >= 10.5 + carries["startX"]))
            | ((carries["startX"] > 60) & (carries["endX"] >= 5.25 + carries["startX"]))
            | ((carries["endX"] >= 83) & (carries["endY"] >= 21) & (carries["endY"] <= 79))
        )
        carries["intoPenArea"] = (
            (carries["endX"] >= 83) & (carries["endY"] >= 21) & (carries["endY"] <= 79)
        )
        carries_agg = (
            carries.groupby("playerId")
            .agg(
                progressive_carrying_distance=("progressive_carrying_distance", "sum"),
                progressive_carries=("isProgressive", "sum"),
                carries_into_penalty_area=("intoPenArea", "sum"),
            )
        )

        player_info = self.players[['playerId', 'playerName', 'teamName', 'minutes', 'position']]
        player_info = player_info.set_index('playerId')

        metrics = pd.concat(
            [player_info, shooting_agg, dribbling_agg, passing_agg_basic,
             passing_agg_advanced, defense_agg, touches_agg, carries_agg,
             xga_agg, xga_op_agg, def_ft_agg],
            axis=1,
        )


        metrics = metrics.drop(index=[-1, 0], errors='ignore').fillna(0.0)

        return metrics
    
    def generate_normalized_metrics(self):
        """
        normalizes applicable metrics to per 90 (figuring out TIP/Otip soon)
        """
        df = self.create_metrics().copy()

        
        df['minutes'] = df['minutes'].astype(str).str.replace(',', '').astype(float)

        exclude = {
        'playerName', 'teamName', 'minutes',
        'completion_percentage', 'forward_pct', 'dribble_success',
        'tackle_success', 'aerial_success',
            }
        cols = [c for c in df.columns
            if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

        nineties = df['minutes'] / 90
        df[cols] = df[cols].div(nineties, axis=0)
        
        return df

    def generate_ratings(self, df_data, position: str, metric_names_and_weights: dict, inclusive: bool = False, minutes_cutoff: int = 200):
        """
        Data input should be normalized

        Metric names and weights should be structured as a dictionary in the below format
        {npxg:30, shots:30}...

        Possible positions: 'FW','MF','DF','GK'
        """

        total = sum(metric_names_and_weights.values())
        if total != 100:
            raise ValueError(f'values must add up to 100. Your values add to {total}')

        demographic_cols = ['playerName', 'teamName', 'position', 'minutes']
        stat_cols = list(metric_names_and_weights.keys())

        cols = demographic_cols + stat_cols

        df = df_data.copy()

        if inclusive == False:
            df = df[df['position'].fillna('').str.contains(position, na=False)]
        else:
            df = df[df['position'] == position]

        df = df[df['minutes'] >= minutes_cutoff]

        df = df[cols]
        df_zscores = df.copy()

        for col in stat_cols:
            df_zscores[f'{col}_zscore'] = zscore(df_zscores[col], nan_policy='omit')

        weights = pd.Series(metric_names_and_weights)

        NEGATIVE_METRICS = {"errors_leading_to_goal", "errors_leading_to_shot",
                            "dribbles_lost", "aerial_losses", "tackles_lost"}

        for col in stat_cols:
            series = -df_zscores[col] if col in NEGATIVE_METRICS else df_zscores[col]
            df_zscores[f'{col}_zscore'] = zscore(series, nan_policy='omit')
        
        zscore_cols = [f'{metric}_zscore' for metric in weights.index]

        df_zscores['rating'] = (
            df_zscores[zscore_cols]
            .mul(weights.values, axis=1)
            .sum(axis=1)
            / weights.sum()
        )

        df_zscores['rating_100'] = (
            norm.cdf(df_zscores['rating']) * 100
        )

        return df_zscores
    
    def _team_possession(self):
        passes = self.events[self.events["type"] == "Pass"]
        match_team = passes.groupby(["matchId", "teamId"]).size().rename("p").reset_index()
        match_team["share"] = match_team["p"] / match_team.groupby("matchId")["p"].transform("sum")
        return match_team.groupby("teamId")["share"].mean()

    def possession_adjust(self, metrics):
        """
        Takes a metrics frame from create_metrics (or normalized) and returns a
        copy with defensive volume columns scaled by a possession-adjustment
        factor. Rates and attacking metrics are left untouched.
        """
        poss = self._team_possession()
        opp_poss = 1 - poss
        factor = opp_poss.mean() / opp_poss   
        
        player_team = (
            self.events.groupby("playerId")["teamId"]
            .agg(lambda s: s.value_counts().index[0])
        )
        pf = player_team.map(factor)

        padj_cols = ["interceptions", "tackles_won", "tackles_lost",
                     "clearances", "recoveries", "blocks", "blocked_crosses",
                     "interceptions_in_box", "def_actions_final_third"]

        out = metrics.copy()
        cols = [c for c in padj_cols if c in out.columns]
        out[cols] = out[cols].mul(pf.reindex(out.index), axis=0)
        return out
    
    def simplify_ratings(self, df_ratings):
        """
        Trims a generate_ratings output down to identifiers, the 0-100 rating,
        and the per-metric z-scores. Sorted by rating descending.
        """
        id_cols = ["playerName", "teamName", "minutes"]
        zscore_cols = [c for c in df_ratings.columns if c.endswith("_zscore")]
        keep = [c for c in id_cols if c in df_ratings.columns] + ["rating_100"] + zscore_cols 
        return df_ratings[keep].sort_values("rating_100", ascending=False)
    
    def display_table(self, df_simplified, n=10):
        """
        Renders the simplified ratings as a stoplight-formatted table.
        z-score columns get red-yellow-green shading centered at 0, the rating
        gets a green ramp, minutes show as integers, and column headers are
        capitalized with the '_zscore' suffix removed.

        n = number of players to show (default 10).
        """
        df = df_simplified.head(n).copy().reset_index(drop=True)

        zscore_cols = [c for c in df.columns if c.endswith("_zscore")]

        rename = {}
        for c in df.columns:
            label = c[:-7] if c.endswith("_zscore") else c
            rename[c] = label.replace("_", " ").title()
        rename["rating_100"] = "Rating"
        rename["playerName"] = "Player"
        rename["teamName"] = "Team"
        df = df.rename(columns=rename)

        new_zcols = [rename[c] for c in zscore_cols]

        int_cols = [c for c in ["Minutes"] if c in df.columns]
        fmt = {c: "{:.0f}" for c in int_cols}
        fmt.update({
            c: "{:.2f}" for c in df.columns
            if c not in int_cols and df[c].dtype.kind == "f"
        })

        styler = (
            df.style
            .background_gradient(cmap="RdYlGn", subset=new_zcols, vmin=-2, vmax=2)
            .background_gradient(cmap="Greens", subset=["Rating"], vmin=0, vmax=100)
            .format(fmt)
            .hide(axis="index")
            .set_properties(**{"text-align": "center"})
            .set_table_styles([
                {"selector": "th", "props": [("text-align", "center"),
                                             ("font-weight", "bold")]}
            ])
        )
        return styler

        

In [4]:
obj = Metrics('/Users/owner/Desktop/Natabase/mls.duckdb', conditions= 'YEAR(startDate) = 2026', players_tablename= 'players_2026_fbref')
df = obj.generate_normalized_metrics()




/Users/owner/Desktop/GitHub/NextStepsFootballAnallytics/CompRatingArticle/MLSCompRatingArticle/xg_model_baked.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prev = ev.groupby("matchId")["keyPassThroughball"].shift(1).fillna(False).astype(int)
/var/folders/hq/8f5kvgjd5dx23mthw16_z1lc0000gn/T/ipykernel_31192/3046728561.py:89: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ].fillna(False).astype(bool).any(axis=1)


In [43]:
df

,playerName,teamName,minutes,position,shots,np_goals,npxg,big_chances_missed,big_chances_scored,big_chances,...,aerial_success,touches,touches_in_final_third,fouls_won,progressive_carrying_distance,progressive_carries,carries_into_penalty_area,xg_assisted,xg_assisted_op,def_actions_final_third
playerId,,,,,,,,,,,,,,,,,,,,,
413783.0,Paxten Aaronson,Colorado Rapids,1165.0,MF,2.085837,0.309013,0.163326,0.077253,0.231760,0.309013,...,33.333333,48.283262,16.300429,1.467811,12.059227,0.618026,0.077253,0.151798,0.136120,1.622318
412540.0,Liel Abada,Charlotte,363.0,"MF,FW",2.479339,0.000000,0.251025,0.495868,0.000000,0.495868,...,22.222222,54.793388,22.066116,0.495868,23.008264,0.495868,0.247934,0.120084,0.120084,1.735537
572932.0,Ezequiel Abadia-Reda,Inter Miami,39.0,DF,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,50.000000,85.384615,16.153846,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.307692
528754.0,Wessam Abou Ali,Columbus Crew,565.0,FW,3.026549,0.796460,0.499193,0.637168,0.477876,1.115044,...,33.333333,37.592920,8.123894,1.911504,32.591150,0.796460,0.000000,0.020419,0.020419,1.115044
334731.0,Lalas Abubakar,FC Dallas,53.0,DF,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,42.452830,5.094340,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436073.0,Sean Zawadzki,Columbus Crew,1350.0,"DF,MF",0.333333,0.000000,0.020172,0.000000,0.000000,0.000000,...,70.967742,77.200000,7.600000,0.600000,22.000000,0.800000,0.000000,0.004695,0.004695,0.533333
126914.0,Walker Zimmerman,Toronto FC,822.0,DF,1.094891,0.109489,0.093998,0.109489,0.000000,0.109489,...,78.947368,62.518248,8.321168,0.547445,4.281022,0.218978,0.000000,0.000000,0.000000,0.218978
355450.0,Philip Zinckernagel,Chicago Fire,1137.0,MF,3.720317,0.316623,0.481071,0.158311,0.237467,0.395778,...,36.363636,62.770449,26.042216,2.612137,18.934037,0.474934,0.158311,0.220271,0.149649,2.532982


In [5]:
df_padj = obj.possession_adjust(df)

In [31]:
dynamic_midfielder = obj.generate_ratings(position='MF', df_data= df, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'npxg': 5,'xg_assisted':10, 'tackles_won': 5, 'tackle_success': 5, 'interceptions': 5, 'progressive_passing_distance': 15, 'progressive_passes' :15, 'progressive_carrying_distance': 5, 'passes_into_pen_area': 10, 'aerial_success': 5, 'pass_epv': 15, 'touches': 5})
dynamic_midfielder = obj.simplify_ratings(dynamic_midfielder)
dynamic_midfielder = obj.display_table(dynamic_midfielder)
dynamic_midfielder

Player,Team,Minutes,Rating,Npxg,Xg Assisted,Tackles Won,Tackle Success,Interceptions,Progressive Passing Distance,Progressive Passes,Progressive Carrying Distance,Passes Into Pen Area,Aerial Success,Pass Epv,Touches
Sebastian Berhalter,Vancouver W'caps,1121,95.40,0.74,2.53,0.81,-0.46,0.04,1.90,1.08,0.58,2.50,1.10,3.28,2.03
Pedro Soma,San Diego FC,400,95.37,-0.91,0.25,-0.16,0.46,-0.36,3.41,3.18,8.74,0.04,-1.27,1.07,3.57
Jeppe Tverskov,San Diego FC,845,93.16,-0.63,0.32,1.46,0.90,2.18,3.01,0.95,5.70,0.14,0.33,0.88,4.36
Joaquín Pereyra,Minnesota Utd,1277,89.81,0.17,1.24,-0.51,0.75,0.76,1.09,1.36,-0.18,2.37,-0.48,3.09,1.09
Lionel Messi,Inter Miami,1242,89.06,4.57,3.67,-1.12,-0.21,-1.35,0.87,-0.21,0.94,3.35,-0.04,1.70,0.67
Rodrigo De Paul,Inter Miami,1170,87.83,-0.17,2.41,-0.67,1.80,0.40,2.41,0.47,1.38,1.01,-0.39,0.98,2.57
Evander,FC Cincinnati,1109,86.70,2.88,4.03,-0.89,-0.78,-0.86,-0.08,1.03,-0.07,2.89,-0.17,1.73,0.29
Alonso Coello,Toronto FC,1028,86.63,-1.04,0.99,0.42,-0.04,2.65,1.90,-0.19,-0.64,2.11,1.54,2.20,1.35
Dylan Chambost,Columbus Crew,765,83.97,-0.84,0.87,0.54,0.86,0.81,2.11,1.74,1.76,0.82,0.49,-0.23,2.00
Ashley Westwood,Charlotte,1203,82.97,-0.82,0.16,-0.96,0.96,-0.92,2.60,0.16,0.11,1.58,1.37,2.22,0.89


In [32]:
wide_attacker_creative = obj.generate_ratings(position='FW', df_data= df, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'npxg': 15,'xg_assisted_op':35, 'dribble_success': 10, 'dribbles_total': 20, 'shots': 5, 'pass_epv': 5, 'through_balls_completed': 5, 'progressive_carrying_distance': 5}).sort_values('rating_100', ascending = False)
wide_attacker_creative = obj.simplify_ratings(wide_attacker_creative)
wide_attacker_creative = obj.display_table(wide_attacker_creative)
wide_attacker_creative

Player,Team,Minutes,Rating,Npxg,Xg Assisted Op,Dribble Success,Dribbles Total,Shots,Pass Epv,Through Balls Completed,Progressive Carrying Distance
Lionel Messi,Inter Miami,1242,99.73,2.20,4.14,0.50,2.28,4.14,1.70,2.73,1.28
Evander,FC Cincinnati,1109,94.73,1.04,2.05,0.45,1.73,1.64,1.73,3.79,-0.03
Cade Cowell,RB New York,879,88.84,-0.47,3.35,-0.82,0.71,0.33,0.97,-0.69,0.48
Luis Suárez,Inter Miami,614,88.77,1.71,3.04,-1.22,-0.87,1.13,0.70,1.62,0.32
Tyler Boyd,LAFC,448,84.19,-0.86,2.22,0.69,1.31,0.27,-0.01,0.89,-0.70
Miguel Almirón,Atlanta Utd,658,81.96,-0.33,1.91,1.71,-0.06,0.26,1.28,1.46,-0.28
Tomás Chancalay,Minnesota Utd,1114,81.47,-0.70,1.89,-0.15,1.52,0.42,1.62,-0.69,-0.32
Denis Bouanga,LAFC,1196,79.39,0.23,0.85,0.35,1.71,1.24,0.44,0.49,0.08
Wilfried Zaha,Charlotte,1161,77.49,-0.49,1.45,0.77,0.76,-0.23,0.25,1.14,0.69
Gabriel Chaves,LA Galaxy,1211,76.07,0.53,0.25,0.28,1.60,2.37,0.97,-0.11,0.62


In [45]:
wide_attacker_dribbler = obj.generate_ratings(position='FW', df_data= df, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'npxg': 15,'xg_assisted_op':15, 'dribble_success': 20, 'dribbles_total': 20, 'pass_epv': 5, 'progressive_carrying_distance': 15, 'carries_into_penalty_area': 10}).sort_values('rating_100', ascending = False)
wide_attacker_dribbler = obj.simplify_ratings(wide_attacker_dribbler)
wide_attacker_dribbler = obj.display_table(wide_attacker_dribbler)
wide_attacker_dribbler

Player,Team,Minutes,Rating,Npxg,Xg Assisted Op,Dribble Success,Dribbles Total,Pass Epv,Progressive Carrying Distance,Carries Into Penalty Area
Lionel Messi,Inter Miami,1242,96.86,2.20,4.14,0.50,2.28,1.70,1.28,0.77
Anders Dreyer,San Diego FC,1284,88.75,0.77,-0.30,1.08,-0.92,2.27,5.21,2.16
David Vazquez,San Diego FC,740,85.09,-0.04,-0.97,0.49,-0.27,-0.43,4.95,4.27
Justin Ellis,Orlando City,534,84.81,-0.32,0.72,3.11,0.79,-0.34,0.32,1.59
Hugo Picard,Columbus Crew,752,84.56,-0.31,0.19,1.74,0.29,-0.34,2.61,2.56
Evander,FC Cincinnati,1109,81.83,1.04,2.05,0.45,1.73,1.73,-0.03,-0.71
Gabriel Chaves,LA Galaxy,1211,80.76,0.53,0.25,0.28,1.60,0.97,0.62,2.33
Denis Bouanga,LAFC,1196,78.58,0.23,0.85,0.35,1.71,0.44,0.08,1.86
Georgi Minoungou,Seattle Sounders/Colorado Rapids,748,77.13,-0.95,0.18,-0.10,4.38,-0.84,-0.31,0.93
Guilherme,Houston Dynamo,1160,76.57,-0.50,0.41,1.25,1.95,0.48,0.26,0.35


In [ ]:
striker = obj.generate_ratings(position='FW', df_data= df, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'npxg': 30,'xg_assisted_op':5, 'shots': 20, 'np_goals': 20, 'pass_epv': 5, 'aerial_success': 10, 'def_actions_final_third':10 }).sort_values('rating_100', ascending = False)
striker = obj.simplify_ratings(striker)
striker = obj.display_table(striker)
striker

Player,Team,Minutes,Rating,Npxg,Xg Assisted Op,Shots,Np Goals,Pass Epv,Aerial Success,Def Actions Final Third
Lionel Messi,Inter Miami,1242,98.40,2.20,4.14,4.14,1.73,1.70,0.13,0.06
Sam Surridge,Nashville SC,457,98.37,3.72,-1.38,1.33,4.71,-0.91,-0.00,-0.71
Petar Musa,FC Dallas,1040,95.67,2.50,-0.47,2.02,2.00,-0.64,1.05,1.10
Hugo Cuypers,Chicago Fire,989,94.86,3.05,-0.76,1.03,2.51,-0.91,-0.06,0.97
Brian White,Vancouver W'caps,1197,92.75,2.95,-0.50,1.75,1.27,-0.99,0.90,-0.47
Luis Suárez,Inter Miami,614,90.20,1.71,3.04,1.13,2.05,0.70,0.68,-1.13
Klauss,LA Galaxy,546,86.03,1.60,-0.43,1.05,1.84,-0.73,0.98,-0.16
Preston Judd,SJ Earthquakes,1183,83.63,1.24,-0.48,1.63,1.89,-0.65,0.68,-1.08
Kevin Kelsy,Portland Timbers,735,80.76,1.26,0.11,0.64,1.03,-0.69,1.38,0.50
Sergi Solans,Real Salt Lake,872,80.04,1.18,-0.23,1.06,1.05,-0.85,1.00,0.22


In [35]:
ten = obj.generate_ratings(position='MF', df_data= df, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'npxg': 10,'xg_assisted_op':25, 'shots': 5, 'np_goals': 5, 'pass_epv': 20, 'through_balls_completed': 10, 'def_actions_final_third':10, 'dribbles_won': 5, 'dribble_success':5, 'carries_into_penalty_area' : 5}).sort_values('rating_100', ascending = False)
ten = obj.simplify_ratings(ten)
ten = obj.display_table(ten)
ten

Player,Team,Minutes,Rating,Npxg,Xg Assisted Op,Shots,Np Goals,Pass Epv,Through Balls Completed,Def Actions Final Third,Dribbles Won,Dribble Success,Carries Into Penalty Area
Lionel Messi,Inter Miami,1242,99.75,4.57,4.14,4.67,3.75,1.70,3.04,0.34,2.63,0.19,1.45
Timo Werner,SJ Earthquakes,456,99.06,2.78,5.29,0.95,1.38,2.46,1.01,0.55,0.58,-0.35,-0.57
Evander,FC Cincinnati,1109,96.56,2.88,2.14,2.43,1.93,1.73,4.18,-0.66,2.07,0.15,-0.57
Thomas Müller,Vancouver W'caps,842,92.38,1.68,3.28,0.72,0.32,1.29,2.07,0.25,-0.63,-0.72,-0.57
Cade Cowell,RB New York,879,90.76,0.68,3.39,1.25,-0.34,0.88,-0.67,1.76,0.16,-0.87,2.28
Philip Zinckernagel,Chicago Fire,1137,90.68,2.71,1.01,2.33,0.92,1.38,0.68,2.04,1.10,-0.23,0.90
Rodrigo De Paul,Inter Miami,1170,87.77,-0.17,2.84,0.42,0.87,0.98,2.61,-0.00,-0.45,-0.00,-0.57
Cavan Sullivan,Philadelphia Union,551,86.98,0.45,1.09,0.87,0.02,2.26,0.72,1.65,1.83,0.23,-0.57
Diego Luna,Real Salt Lake,605,86.92,0.70,1.85,0.97,2.56,-0.32,3.14,1.57,0.12,0.59,-0.57
Miguel Almirón,Atlanta Utd,658,86.37,0.88,2.01,1.19,-0.94,1.22,1.66,-0.00,1.10,1.16,-0.57


In [36]:
destructive_midfielder = obj.generate_ratings(position='MF', df_data= df, inclusive = True, minutes_cutoff= 400, metric_names_and_weights={'tackles_won': 20, 'tackle_success': 20, 'interceptions': 20, 'progressive_passing_distance': 15, 'progressive_passes' :15, 'aerial_success': 10}).sort_values('rating_100', ascending = False)
destructive_midfielder= obj.simplify_ratings(destructive_midfielder)
destructive_midfielder = obj.display_table(destructive_midfielder)
destructive_midfielder

Player,Team,Minutes,Rating,Tackles Won,Tackle Success,Interceptions,Progressive Passing Distance,Progressive Passes,Aerial Success
Andrés Cubas,Vancouver W'caps,860,96.09,4.06,0.01,2.62,0.28,1.62,1.38
Jeppe Tverskov,San Diego FC,845,91.99,1.43,1.07,1.89,2.71,0.61,0.29
Yannick Bright,Inter Miami,957,88.73,2.32,-0.43,2.51,0.49,0.71,1.53
Tristan Muyumba,Atlanta Utd,1018,87.07,2.22,1.51,0.73,1.15,0.24,0.29
Sékou Tidiany Bangoura,Columbus Crew,459,84.11,0.95,-0.28,1.77,0.11,2.71,0.86
Dylan Chambost,Columbus Crew,765,81.95,0.41,1.03,0.51,1.76,1.43,0.46
Stijn Spierings,Real Salt Lake,752,81.91,1.17,1.05,0.78,0.68,0.41,1.50
Stephen Eustáquio,LAFC,601,81.29,0.81,0.08,0.32,1.13,2.07,1.65
José Caicedo,Portland Timbers,503,81.10,1.74,0.23,0.77,1.14,0.47,0.92
Alonso Coello,Toronto FC,1028,79.97,0.28,0.05,2.37,1.54,-0.58,1.57


In [ ]:
aggressive_cb = obj.generate_ratings(position='DF', df_data= df_padj, inclusive = True, minutes_cutoff= 400, metric_names_and_weights={'tackles_won': 15, 'tackle_success': 20, 'interceptions': 5, 'clearances' : 5, 'aerial_success': 20, 'aerials' : 10, 'completion_percentage': 5, 'errors_leading_to_shot' : 15, 'progressive_passing_distance': 5 }).sort_values('rating_100', ascending = False)
aggressive_cb = obj.simplify_ratings(aggressive_cb)s11
aggressive_cb = obj.display_table(aggressive_cb)
aggressive_cb

Player,Team,Minutes,Rating,Tackles Won,Tackle Success,Interceptions,Clearances,Aerial Success,Aerials,Completion Percentage,Errors Leading To Shot,Progressive Passing Distance
Jackson Ragen,Seattle Sounders,1059,78.40,-0.36,1.85,-0.98,1.53,1.26,0.51,1.14,-0.30,2.56
Nicolás Romero,Minnesota Utd,638,76.72,-0.41,2.53,0.29,-0.64,1.55,-0.71,-0.21,0.67,-0.53
Carlos Garcés,LA Galaxy,704,76.69,3.04,1.06,0.99,-0.24,-0.48,-0.28,0.75,0.67,0.16
Micael,Inter Miami,1260,75.29,1.24,1.30,1.05,-0.23,0.78,-0.42,1.32,-0.14,0.76
Guilherme Biro,Austin FC,854,73.95,-0.18,1.30,-0.20,-0.15,1.06,1.91,-0.60,0.67,-0.97
Morrison Agyemang,Charlotte,1215,73.90,-0.47,1.09,0.90,0.33,0.59,1.24,0.95,0.67,0.85
Miles Robinson,FC Cincinnati,603,73.75,0.12,0.51,-0.97,0.52,1.14,1.48,-0.20,0.67,1.44
Osaze Urhoghide,FC Dallas,1079,73.31,-0.40,0.73,0.73,1.48,1.19,2.33,0.68,-0.28,-0.76
Nolan Norris,FC Dallas,899,72.90,0.78,1.11,1.01,0.71,0.00,1.22,-0.75,0.67,-0.03
Nathan Harriel,Philadelphia Union,1350,72.51,0.76,-0.28,3.68,-0.26,1.14,3.24,-2.65,-0.09,-0.74


In [ ]:
progressive_cb  = obj.generate_ratings(position='DF', df_data= df_padj, inclusive = True, minutes_cutoff= 400, metric_names_and_weights={'tackle_success': 10, 'errors_leading_to_shot': 10, 'aerial_success': 10, 'completion_percentage': 5, 'progressive_passes': 310,'progressive_carrying_distance': 10, 'progressive_passing_distance': 25 }).sort_values('rating_100', ascending = False)
progressive_cb = obj.simplify_ratings(progressive_cb)
progressive_cb = obj.display_table(progressive_cb)
progressive_cb

Player,Team,Minutes,Rating,Tackle Success,Errors Leading To Shot,Aerial Success,Completion Percentage,Progressive Passes,Progressive Carrying Distance,Progressive Passing Distance
Jackson Ragen,Seattle Sounders,1059,89.63,1.85,-0.30,1.26,1.14,1.03,-0.26,2.56
Jeison Palacios,Nashville SC,978,85.88,0.55,0.67,-0.66,1.30,1.85,-0.02,1.61
Alex Roldan,Seattle Sounders,739,84.31,-0.04,0.67,0.65,-0.13,2.60,-0.71,0.71
Christopher McVey,San Diego FC,1053,84.06,-0.09,-0.30,-0.75,1.13,-0.60,4.76,3.03
Lukas Engel,Real Salt Lake,659,83.00,1.18,0.67,0.18,-0.83,1.37,-0.23,1.62
Rudy Camacho,Columbus Crew,1267,82.36,-0.84,0.67,-0.64,0.97,1.04,0.49,2.40
Micael,Inter Miami,1260,82.07,1.30,-0.14,0.78,1.32,1.51,0.14,0.76
Manu Duah,San Diego FC,1199,81.60,0.55,-1.04,-0.62,1.34,0.47,2.85,2.07
Kieran Sargeant,San Diego FC,439,78.17,-3.08,0.67,-2.38,-0.22,4.22,3.64,-1.45
Kamal Miller,Portland Timbers,558,76.42,-1.96,0.67,1.14,1.03,0.38,-0.75,2.58


In [40]:
attacking_fb = obj.generate_ratings(position='DF', df_data= df_padj, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'tackle_success': 15, 'interceptions': 10, 'aerial_success': 10, 'completion_percentage': 5, 'progressive_passes': 10,'progressive_carrying_distance': 10, 'progressive_passing_distance': 10, 'passes_into_pen_area': 10, 'pass_epv' : 10, 'xg_assisted_op': 10}).sort_values('rating_100', ascending = False)
attacking_fb = obj.simplify_ratings(attacking_fb)
attacking_fb = obj.display_table(attacking_fb)
attacking_fb

Player,Team,Minutes,Rating,Tackle Success,Interceptions,Aerial Success,Completion Percentage,Progressive Passes,Progressive Carrying Distance,Progressive Passing Distance,Passes Into Pen Area,Pass Epv,Xg Assisted Op
Francis Westfield,Philadelphia Union,890,87.15,0.39,0.92,1.66,-3.89,-1.00,-0.23,-0.15,3.95,4.74,2.80
Keegan Rosenberry,Colorado Rapids,418,83.57,0.62,1.11,-0.07,-0.32,1.43,0.26,-0.25,2.59,-0.93,4.85
Ben Bender,Philadelphia Union,435,81.49,-1.16,1.59,1.18,-1.74,-0.41,-0.79,-0.54,3.92,4.77,1.87
Christopher McVey,San Diego FC,1053,81.23,-0.11,1.95,-0.55,1.23,-0.61,4.77,2.99,-0.59,1.27,-0.82
Andy Najar,Nashville SC,753,80.88,0.29,0.31,-0.31,-0.17,1.42,1.10,0.81,2.32,-0.34,3.08
Luca Bombino,San Diego FC,931,79.68,0.07,-0.32,0.04,0.03,-0.49,6.38,-0.28,0.71,1.57,0.57
Tayvon Gray,NYCFC,1213,76.73,0.94,0.41,-0.80,0.06,0.91,0.57,0.39,0.40,1.37,2.62
Lukas Engel,Real Salt Lake,659,76.68,1.23,0.69,0.31,-0.65,1.09,-0.26,1.72,0.28,2.72,-0.77
Kai Trewin,NYCFC,1254,76.08,0.58,0.32,0.66,0.61,3.61,0.53,0.78,0.25,-0.09,-0.15
Alex Roldan,Seattle Sounders,739,74.83,-0.05,1.64,0.73,0.01,2.16,-0.74,0.90,0.65,1.22,0.20


In [42]:
balanced_fb = obj.generate_ratings(position='DF', df_data= df_padj, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'tackle_success': 15, 'tackles_won': 20, 'interceptions': 10, 'aerial_success': 20, 'pass_epv': 10, 'progressive_passing_distance': 5, 'errors_leading_to_shot': 10, 'recoveries': 10}).sort_values('rating_100', ascending = False)
balanced_fb = obj.simplify_ratings(balanced_fb)
balanced_fb = obj.display_table(balanced_fb)
balanced_fb

Player,Team,Minutes,Rating,Tackle Success,Tackles Won,Interceptions,Aerial Success,Pass Epv,Progressive Passing Distance,Errors Leading To Shot,Recoveries
Raheem Edwards,Toronto FC,737,92.45,0.17,4.21,1.70,0.85,0.60,-1.00,0.66,1.53
Francis Westfield,Philadelphia Union,890,89.30,0.39,0.82,0.92,1.66,4.74,-0.15,0.66,0.63
Ben Bender,Philadelphia Union,435,83.08,-1.16,0.29,1.59,1.18,4.77,-0.54,0.66,1.65
Alex Roldan,Seattle Sounders,739,81.56,-0.05,1.41,1.64,0.73,1.22,0.90,0.66,0.82
Carlos Garcés,LA Galaxy,704,81.42,1.11,2.66,1.03,-0.31,-0.16,0.41,0.66,0.83
Miki Yamane,LA Galaxy,724,77.99,1.76,3.30,0.68,-0.07,0.46,-1.01,-0.84,-1.18
Mathías Laborda,Vancouver W'caps,1102,76.63,1.07,1.78,0.55,0.56,0.02,0.44,-1.31,1.50
Christopher McVey,San Diego FC,1053,75.80,-0.11,0.42,1.95,-0.55,1.27,2.99,-0.37,3.08
Micael,Inter Miami,1260,75.47,1.36,1.04,1.08,0.85,-0.90,0.95,-0.20,0.63
Nathan Harriel,Philadelphia Union,1350,74.92,-0.30,0.60,3.64,1.18,0.65,-0.40,-0.14,-0.32
